In [ ]:
!pip uninstall -y mediapipe
!pip install mediapipe==0.10.9
!pip install opencv-python-headless numpy scipy scikit-learn filterpy pywavelets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 5.9 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Uninstalling protobuf-5.29.4:
      Successfully uninstalled protobuf-5.29.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-metadata 1.16.1 requires protobuf<6.0.0dev,>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
grpcio-status 1.71.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 3.20.3 which is incompatible.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 44.8 MB/s eta 0:00:00
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110458 sha256=1044dcfe4fa3c7db4f04b1427b94c0fabe0608255c999b2c0b1628c8cab8d97f
  Stored in directory: /root/.cache/pip/wheels/12/dc/3c/e12983eac132d00f82a20c6cbe7b42ce6e96190ef8fa2d15e1
Successfully built filterpy


In [ ]:
import cv2
import mediapipe as mp
import numpy as np

mp_face_mesh = mp.solutions.face_mesh

def get_face_roi(frame, roi_area="forehead"):
    """
    Detects face landmarks using MediaPipe FaceMesh and extracts a region of interest (ROI)
    such as the forehead or cheek area.
    """
    with mp_face_mesh.FaceMesh(
        static_image_mode=False,
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as face_mesh:

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(frame_rgb)

        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                h, w, _ = frame.shape
                if roi_area == "forehead":
                    # Example indices for forehead region
                    roi_indices = [10, 151, 9, 337, 338]
                elif roi_area == "cheek":
                    # Example indices for cheek region
                    roi_indices = [234, 93, 132, 58, 172]
                else:
                    raise ValueError("Invalid ROI area. Choose 'forehead' or 'cheek'.")

                # Extract ROI points (x,y) in image coordinates
                roi_points = [(int(face_landmarks.landmark[i].x * w),
                               int(face_landmarks.landmark[i].y * h)) for i in roi_indices]

                # Create a bounding box around the ROI
                x_min = min(p[0] for p in roi_points)
                x_max = max(p[0] for p in roi_points)
                y_min = min(p[1] for p in roi_points)
                y_max = max(p[1] for p in roi_points)

                # Crop the ROI
                roi = frame[y_min:y_max, x_min:x_max]

                return roi  # Return the first detected face's ROI
    return None  # If no face detected


In [ ]:
import pywt
from sklearn.decomposition import FastICA
from scipy.signal import butter, filtfilt

def skin_segmentation_ycrcb_hsv(image):
    """
    Uses Hybrid YCrCb + HSV for Skin Segmentation.
    Returns a mask-applied image where only skin pixels remain.
    """
    if image is None or image.size == 0:
        return None

    # Convert to YCrCb and HSV
    ycrcb = cv2.cvtColor(image, cv2.COLOR_BGR2YCrCb)
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Skin color range in YCrCb
    ycrcb_lower = np.array([0, 133, 77], dtype=np.uint8)
    ycrcb_upper = np.array([255, 173, 127], dtype=np.uint8)

    # Skin color range in HSV
    hsv_lower = np.array([0, 40, 30], dtype=np.uint8)
    hsv_upper = np.array([43, 255, 254], dtype=np.uint8)

    # Thresholding
    mask_ycrcb = cv2.inRange(ycrcb, ycrcb_lower, ycrcb_upper)
    mask_hsv = cv2.inRange(hsv, hsv_lower, hsv_upper)

    # Combine both masks
    final_mask = cv2.bitwise_and(mask_ycrcb, mask_hsv)

    return cv2.bitwise_and(image, image, mask=final_mask)

def wavelet_denoise(signal):
    """
    Applies Wavelet Denoising to 1D signal.
    """
    coeffs = pywt.wavedec(signal, 'db4', level=3)
    threshold = np.std(coeffs[-1])  # Estimate noise threshold from detail coefficients
    new_coeffs = [pywt.threshold(c, threshold, mode='soft') for c in coeffs]
    return pywt.waverec(new_coeffs, 'db4')

def apply_ica(rgb_signals):
    """
    Applies ICA for noise reduction and extracting rPPG signal.
    Expects shape (N, 3) or (N, 1).
    """
    ica = FastICA(n_components=1, random_state=42)
    ica_output = ica.fit_transform(rgb_signals)
    return ica_output.flatten()

def butterworth_filter(signal, fs=30, lowcut=0.7, highcut=3.0, order=4):
    """
    Applies Butterworth Bandpass Filter to isolate typical heart rate frequencies (42-180 BPM).
    """
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal)


In [ ]:
from google.colab.patches import cv2_imshow

def process_video(video_path, roi_area="forehead", show_every_n_frames=30):
    """
    Reads a video, extracts ROI using FaceMesh, applies skin segmentation,
    and processes the rPPG signal with wavelet denoising, ICA, and Butterworth filtering.
    Returns an array of processed signals for each frame.
    """
    cap = cv2.VideoCapture(r'/content/sample - Made with Clipchamp (1).mp4')

    if not cap.isOpened():
        print("Error: Unable to open video file.")
        return None

    frame_count = 0
    rppg_signals = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break  # End of video

        frame_count += 1

        # 1. Extract ROI (e.g., forehead)
        roi_frame = get_face_roi(frame, roi_area=roi_area)
        if roi_frame is None or roi_frame.size == 0:
            # If no face detected or ROI is invalid, skip
            continue

        # 2. Skin segmentation
        segmented_skin = skin_segmentation_ycrcb_hsv(roi_frame)
        if segmented_skin is None:
            continue

        # 3. Flatten to Nx3 for rPPG (RGB) or Nx1 for single channel
        rgb_values = segmented_skin.reshape(-1, 3)
        if rgb_values.shape[0] == 0:
            # No pixels found, skip
            continue

        # 4. Extract a 1D signal (e.g., green channel) or keep all 3 channels
        # Let's do a 3-channel approach for ICA
        # Then we'll apply wavelet & Butterworth to the ICA output
        # Optionally, you could just do green_channel = rgb_values[:,1]
        # and skip ICA if you want a simpler pipeline.
        try:
            # Wavelet denoise each channel
            denoised_r = wavelet_denoise(rgb_values[:, 0])
            denoised_g = wavelet_denoise(rgb_values[:, 1])
            denoised_b = wavelet_denoise(rgb_values[:, 2])

            # Stack for ICA (N,3)
            denoised_rgb = np.column_stack((denoised_r, denoised_g, denoised_b))

            # Apply ICA
            ica_signal = apply_ica(denoised_rgb)

            # Bandpass filter
            filtered_signal = butterworth_filter(ica_signal, fs=30)

            rppg_signals.append(filtered_signal)
        except Exception as e:
            print(f"Frame {frame_count}: Error during signal processing - {e}")
            continue

        # (Optional) Show the segmented skin every N frames
        if frame_count % show_every_n_frames == 0:
            print(f"Processing frame {frame_count}")
            cv2_imshow(segmented_skin)

    cap.release()
    print(f"Processed {frame_count} frames.")

    # Stack signals into one 2D array (num_frames, signal_length)
    # The signals may have slightly different lengths due to wavelet transform, etc.
    # We'll handle it by padding or just keep them as objects.
    # A simpler approach is to keep them as a Python list if length can vary.
    # If all signals are the same length, you can stack them:
    # e.g. np.vstack(rppg_signals)

    return rppg_signals


In [ ]:
rppg_signals=process_video(r'/content/sample - Made with Clipchamp (1).mp4')

Processed 13 frames.


In [ ]:
rppg_signals

[array([-0.01126706, -0.0268283 , -0.042986  , ...,  0.01607983,
         0.00418675, -0.01116806]),
 array([-0.01779689, -0.03031209, -0.04325378, ...,  0.0402017 ,
         0.01471078, -0.01815349]),
 array([-0.00302083,  0.00279178,  0.0087479 , ...,  0.02917405,
         0.02620771,  0.02366449]),
 array([ 0.01039186,  0.00488906, -0.00080169, ..., -0.03055204,
        -0.02955621, -0.03073821]),
 array([-0.00060887, -0.00051979, -0.00041158, ..., -0.01568925,
         0.00651105,  0.03506535]),
 array([-7.94330877e-11, -7.41054417e-11, -6.66159849e-11, ...,
        -3.69664755e-02, -1.63173299e-02,  9.97733412e-03]),
 array([-0.0012355 , -0.00072252, -0.00015299, ..., -0.04710917,
        -0.02143048,  0.00734337]),
 array([-0.00133163, -0.00080612, -0.00022112, ..., -0.0323424 ,
        -0.01395383,  0.00689179]),
 array([-0.00266443,  0.00237575,  0.00751274, ..., -0.01752985,
        -0.00791061,  0.0024247 ]),
 array([-0.00127781,  0.00310986,  0.00764746, ..., -0.04196092,
  